## 3.3.5 文本生成

对应 PDF **Problem (decoding)**：实现一个解码函数，支持提示词续写、最大生成长度、
温度缩放与 top-p（核）采样。

核心是一个循环：**每轮跑一次前向 → 取最后一个位置的分布 → 采样出一个 token → 拼回输入末尾**，直到撞上 `<|endoftext|>` 或到达长度上限。

In [ ]:
import os
import sys

# 前面几节实现的模块统一放在项目根的 scripts/ 下，按相对位置把它加进搜索路径
for path in [os.path.abspath("."), os.path.abspath("../../../scripts")]:
    if path not in sys.path:
        sys.path.insert(0, path)

In [ ]:
import torch

def decode(model, prompt, eot_token_id=None, max_num_generate_token=256,
           temperature=0.5, top_p=0.9, context_length=256):
    '''
    从语言模型采样生成，返回新生成的 token ID 列表（不含 prompt 自身）

    model: torch.nn.Module  语言模型，输入 (batch, seq_len) 输出 (batch, seq_len, vocab_size)
    prompt: list[int]  用户给的提示词，已编码成 token ID
    eot_token_id: int  结束符 <|endoftext|> 的 ID，生成到它就停；None 表示不提前停
    max_num_generate_token: int  生成 token 的总数上限（含 prompt 长度）
    temperature: float  τ，越小分布越尖锐（趋近贪心），越大越平缓
    top_p: float  核采样阈值，只保留累计概率达到 p 的那一小撮候选
    context_length: int  模型能接受的最大序列长度
    '''
    output = []
    x = torch.tensor(prompt).unsqueeze(0)            # (1, seq_len) 补上 batch 维
    model.eval()
    torch.set_grad_enabled(False)                    # 生成只做前向，不建图

    # 每轮产出一个 token：前向 → 取末位分布 → 采样 → 拼回输入
    for i in range(max(0, max_num_generate_token - len(prompt))):
        x_input = x[:, -context_length:]             # 输入可能已超过模型的上下文窗口，只留最后一段
        pred_y = model(x_input)[:, -1, :]            # (1, vocab_size) 只要最后一个位置

        # 温度缩放：τ→0 时最大的 logit 一枝独秀（趋近贪心），τ 越大分布越平、越敢选低概率词
        probs = torch.softmax(pred_y / temperature, dim=-1).squeeze(0)   # (vocab_size,)

        # top-p：先按概率从大到小排序并求前缀和，累计到 p 的那一段就是要保留的"核"
        sorted_probs, sorted_indices = probs.sort(descending=True)
        cumsum = sorted_probs.cumsum(dim=-1)
        cutoff_idx = torch.searchsorted(cumsum, top_p)   # 第一个使前缀和 >= p 的下标
        sorted_mask = torch.arange(cumsum.shape[-1], device=probs.device) <= cutoff_idx

        # 注意这里的下标换算：mask 是按"排序后的顺序"算出来的，必须先经 sorted_indices
        # 映射回原始下标，才能作用到原始顺序的 probs 上；~sorted_mask 挑出的正是要丢掉的尾部
        probs[sorted_indices[~sorted_mask]] = 0

        probs = probs / probs.sum()                  # 丢掉尾部之后要重新归一化
        next_token_id = torch.multinomial(probs, num_samples=1).item()

        output.append(next_token_id)
        x = torch.cat([x, torch.tensor([[next_token_id]], device=x.device)], dim=1)
        if eot_token_id is not None and next_token_id == eot_token_id:
            break

    return output

In [ ]:
import torch

# ---------- 例子 1：温度 τ 怎么改变采样分布 ----------
logits = torch.tensor([2.0, 1.0, 0.5, 0.2, -1.0])   # 5 个候选 token 的 logits

for tau in [0.1, 0.5, 1.0, 2.0]:
    p = torch.softmax(logits / tau, dim=-1)
    print(f"τ={tau:<4} → 概率 {[round(v, 3) for v in p.tolist()]}")
# τ=0.1  → [1.0,   0.0,   0.0,   0.0,   0.0  ]   ← 几乎 one-hot，等价于贪心
# τ=1.0  → [0.554, 0.204, 0.124, 0.092, 0.028]   ← 原始 softmax
# τ=2.0  → [0.369, 0.224, 0.174, 0.150, 0.082]   ← 明显被拉平，尾部候选机会变大

# ---------- 例子 2：top-p 保留多少个候选 ----------
probs = torch.softmax(logits, dim=-1)               # 以 τ=1 的分布为例
for top_p in [0.5, 0.9, 1.0]:
    sorted_probs, _ = probs.sort(descending=True)
    cumsum = sorted_probs.cumsum(dim=-1)
    cutoff = torch.searchsorted(cumsum, top_p).item()
    print(f"top_p={top_p} → 保留前 {cutoff + 1} 个候选，其累计概率 {cumsum[cutoff]:.3f}")
# top_p=0.5 → 保留前 1 个候选，其累计概率 0.554
# top_p=0.9 → 保留前 4 个候选，其累计概率 0.972
# top_p=1.0 → 保留前 5 个候选，其累计概率 1.000   ← 等于不做截断

In [ ]:
import torch


class 接龙模型(torch.nn.Module):
    '''一个假的"语言模型"：永远预测"下一个 token = 当前最后一个 token + 1"。

    它不学任何东西，纯粹用来看清解码循环在做什么——真模型跑出来的分布看不出规律，
    这里换成确定规则后，temperature 与 top-p 的效果能一眼看出来。
    '''
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size

    def forward(self, tokens):
        batch, seq_len = tokens.shape
        logits = torch.full((batch, seq_len, self.vocab_size), -10.0)
        next_id = (tokens[:, -1] + 1) % self.vocab_size      # (batch,) 每个序列的"下一个数"
        logits[torch.arange(batch), :, next_id] = 10.0       # 给正确候选一个远高于其他的分数
        return logits


# ---------- 例子 3：完整解码循环 ----------
model = 接龙模型(vocab_size=64)

# τ 越大，模型越"敢"偏离它最有把握的那个候选，输出随之开始走形
for tau in [0.01, 3.0, 10.0]:
    out = decode(model, prompt=[10], eot_token_id=None, max_num_generate_token=12,
                 temperature=tau, top_p=1.0, context_length=32)
    print(f"τ={tau:<5} → {out}")
# 示例输出（未固定采样种子，每次略有不同）：
# τ=0.01 → [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]   ← 完全按规则走
# τ=3.0  → [11, 12, 13, 14, 27, 28, 29, 30, 31, 32, 11]   ← 中途跑偏一次，之后还能接回来
# τ=10.0 → [42, 10, 54, 21, 55, 56, 57, 19, 28, 36, 44]   ← 基本乱走

# 到点自动停：命中结束符就退出循环
out = decode(model, prompt=[10], eot_token_id=15, max_num_generate_token=12,
             temperature=0.01, top_p=1.0, context_length=32)
print(f"设 eot_token_id=15 → {out}")   # [11, 12, 13, 14, 15] 撞上 15 即停

In [ ]:
import torch
from transformer import TransformerLM

# ---------- 例子 4：换成真正的 TransformerLM，接口完全一致 ----------
# 这里用的是随机初始化的权重，所以输出是乱码；接上 3.3.3 训练出的检查点就能出流畅文本
torch.manual_seed(0)
vocab_size, context_length, d_model, num_layers, num_heads = 64, 32, 32, 2, 4
d_ff = int(d_model * 8 / 3 + 63) // 64 * 64

model = TransformerLM(vocab_size=vocab_size, context_length=context_length,
                      d_model=d_model, num_layers=num_layers, num_heads=num_heads,
                      d_ff=d_ff, rope_theta=10000.0)

out = decode(model, prompt=[10, 20, 30], eot_token_id=0, max_num_generate_token=16,
             temperature=0.8, top_p=0.9, context_length=context_length)
print("生成的 token ID:", out)
# 真实场景里，把这段 ID 交给分词器的 decode 方法就能还原成文本：
#   text = tokenizer.decode(out)